## Setup

In [ ]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from bicep.analysis import BicepResults
from utils.sampling import (
    PanelUtilizationDistribution,
    PvSizingDistribution,
    PanelUpgradeCostDistribution
)

print("Environment setup complete!")

## Why Use Distributions?

Infrastructure costs vary based on:

1. **Building characteristics** - Size, age, existing equipment
2. **Regional variations** - Labor costs, material prices
3. **Technical factors** - Complexity of upgrades, site conditions
4. **Market factors** - Supply chain, contractor availability

Instead of using a single cost estimate, BICEP uses probability distributions to capture this uncertainty.

## Built-in Distributions in BICEP

### 1. Panel Utilization Distribution

Represents how fully existing electrical panels are utilized.

In [ ]:
# Create panel utilization distribution
panel_util = PanelUtilizationDistribution()

# Sample from the distribution
samples = panel_util.constrained_samples(n=5000, min_value=0.1, max_value=0.95)

# Visualize
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=samples,
    nbinsx=50,
    name='Panel Utilization',
    opacity=0.7
))

fig.update_layout(
    title='Panel Utilization Distribution',
    xaxis_title='Utilization Rate',
    yaxis_title='Frequency',
    height=400
)

fig.show()

print(f"Panel Utilization Statistics:")
print(f"  Mean: {np.mean(samples):.3f}")
print(f"  Median: {np.median(samples):.3f}")
print(f"  Std Dev: {np.std(samples):.3f}")
print(f"  Min: {np.min(samples):.3f}")
print(f"  Max: {np.max(samples):.3f}")

### 2. PV Sizing Distribution

Represents the relationship between building loads and solar PV system size.

In [ ]:
# Create PV sizing distribution
pv_sizing = PvSizingDistribution()

# Sample from the distribution
pv_samples = pv_sizing.constrained_samples(n=5000, min_value=0.5, max_value=3.0)

# Visualize
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=pv_samples,
    nbinsx=50,
    name='PV Sizing Ratio',
    opacity=0.7
))

fig.update_layout(
    title='PV System Sizing Distribution\n(PV Capacity / Building Load)',
    xaxis_title='Sizing Ratio',
    yaxis_title='Frequency',
    height=400
)

fig.show()

print(f"PV Sizing Statistics:")
print(f"  Mean: {np.mean(pv_samples):.3f}")
print(f"  Median: {np.median(pv_samples):.3f}")
print(f"  Std Dev: {np.std(pv_samples):.3f}")

### 3. Panel Upgrade Cost Distribution

Represents the variation in electrical panel upgrade costs based on building type and regional factors.

In [ ]:
# Create residential panel upgrade cost distribution
res_cost_dist = PanelUpgradeCostDistribution(residential=True)

# Create commercial panel upgrade cost distribution
com_cost_dist = PanelUpgradeCostDistribution(residential=False)

# Sample from both distributions
res_costs = res_cost_dist.constrained_samples(n=5000, min_value=0, max_value=50000)
com_costs = com_cost_dist.constrained_samples(n=5000, min_value=0, max_value=100000)

# Visualize comparison
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=res_costs,
    nbinsx=50,
    name='Residential',
    opacity=0.7
))

fig.add_trace(go.Histogram(
    x=com_costs,
    nbinsx=50,
    name='Commercial',
    opacity=0.7
))

fig.update_layout(
    title='Panel Upgrade Cost Distribution',
    xaxis_title='Upgrade Cost ($)',
    yaxis_title='Frequency',
    barmode='overlay',
    height=400
)

fig.update_xaxes(tickformat='$,.0f')
fig.show()

print(f"\nResidential Panel Upgrade Cost Statistics:")
print(f"  Mean: ${np.mean(res_costs):,.0f}")
print(f"  Median: ${np.median(res_costs):,.0f}")
print(f"  Std Dev: ${np.std(res_costs):,.0f}")
print(f"  5th percentile: ${np.percentile(res_costs, 5):,.0f}")
print(f"  95th percentile: ${np.percentile(res_costs, 95):,.0f}")

print(f"\nCommercial Panel Upgrade Cost Statistics:")
print(f"  Mean: ${np.mean(com_costs):,.0f}")
print(f"  Median: ${np.median(com_costs):,.0f}")
print(f"  Std Dev: ${np.std(com_costs):,.0f}")
print(f"  5th percentile: ${np.percentile(com_costs, 5):,.0f}")
print(f"  95th percentile: ${np.percentile(com_costs, 95):,.0f}")

## Cost Uncertainty Ranges

Let's explore the range of possible total costs due to distribution variability.

In [ ]:
# Run BICEP analysis
results = BicepResults(scenario='high')

# The total cost reflects the sampled costs from distributions
total_cost = results.aggregated['cost'].sum()

print(f"\nBICEP Analysis Results:")
print(f"Scenario: High")
print(f"Total Annualized Cost: ${total_cost:,.0f}")

# Show cost distribution in results
cost_stats = results.residential['cost'].describe()
print(f"\nCost Statistics (Residential):")
print(cost_stats)

## Understanding Cost Variation

Let's analyze how costs vary across buildings and what factors drive variation.

In [ ]:
# Get cost distribution by building type
residential = results.residential

# Buildings with costs vs without
buildings_with_upgrades = residential[residential['cost'] > 0]
buildings_no_upgrades = residential[residential['cost'] == 0]

print(f"\nBuildings requiring upgrades: {len(buildings_with_upgrades):,} ({len(buildings_with_upgrades)/len(residential)*100:.1f}%)")
print(f"Buildings with no upgrades: {len(buildings_no_upgrades):,} ({len(buildings_no_upgrades)/len(residential)*100:.1f}%)")

# Cost distribution for buildings with upgrades
print(f"\nUpgrade Cost Distribution (buildings with costs only):")
print(buildings_with_upgrades['cost'].describe())

In [ ]:
# Visualize cost distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('All Buildings', 'Buildings with Upgrades Only')
)

# All buildings
fig.add_trace(
    go.Histogram(x=residential['cost'], nbinsx=50, name='All', opacity=0.7),
    row=1, col=1
)

# Buildings with upgrades
fig.add_trace(
    go.Histogram(x=buildings_with_upgrades['cost'], nbinsx=50, name='With Upgrades', opacity=0.7),
    row=1, col=2
)

fig.update_xaxes(title_text='Cost ($)', tickformat='$,.0f', row=1, col=1)
fig.update_xaxes(title_text='Cost ($)', tickformat='$,.0f', row=1, col=2)
fig.update_yaxes(title_text='Number of Buildings')

fig.update_layout(
    title_text='Distribution of Infrastructure Upgrade Costs (Residential)',
    height=500,
    showlegend=False
)

fig.show()

## Cost Drivers and Distributions

In [ ]:
# Analyze which technology drivers influence costs
# Group buildings by which technologies trigger upgrades

residential_with_costs = residential[residential['cost'] > 0]

# Count buildings needing upgrades by technology
ev_drivers = residential_with_costs[residential_with_costs['ev_req_capacity_amp'] > 0]
hp_drivers = residential_with_costs[residential_with_costs['hp_req_capacity_amp'] > 0]
hpwh_drivers = residential_with_costs[residential_with_costs['hpwh_req_capacity_amp'] > 0]
pv_drivers = residential_with_costs[residential_with_costs['pv_req_capacity_amp'] > 0]

print("Buildings with infrastructure needs by technology:")
print(f"  EV Charging: {len(ev_drivers):,} ({len(ev_drivers)/len(residential_with_costs)*100:.1f}%)")
print(f"  Heat Pump: {len(hp_drivers):,} ({len(hp_drivers)/len(residential_with_costs)*100:.1f}%)")
print(f"  HPWH: {len(hpwh_drivers):,} ({len(hpwh_drivers)/len(residential_with_costs)*100:.1f}%)")
print(f"  Solar PV: {len(pv_drivers):,} ({len(pv_drivers)/len(residential_with_costs)*100:.1f}%)")

# Average costs by technology driver
print(f"\nAverage cost by technology driver:")
print(f"  EV Charging: ${ev_drivers['cost'].mean():,.0f}")
print(f"  Heat Pump: ${hp_drivers['cost'].mean():,.0f}")
print(f"  HPWH: ${hpwh_drivers['cost'].mean():,.0f}")
print(f"  Solar PV: ${pv_drivers['cost'].mean():,.0f}")

## Designing Custom Distributions

If you need to use custom cost distributions, consider these approaches.

In [ ]:
# Example: Creating a custom cost distribution

# Approach 1: Empirical Distribution (from your data)
def create_empirical_distribution(cost_data):
    """
    Create an empirical distribution from observed cost data.
    
    Args:
        cost_data: array of observed costs
    
    Returns:
        A function that samples from the empirical distribution
    """
    sorted_costs = np.sort(cost_data)
    
    def sample(n=1):
        return np.random.choice(sorted_costs, size=n, replace=True)
    
    return sample

# Example: Parametric Distribution (log-normal)
def create_lognormal_distribution(mu, sigma, min_val=0, max_val=np.inf):
    """
    Create a log-normal distribution for costs.
    
    Log-normal distributions are common for costs (always positive, right-skewed).
    
    Args:
        mu: mean of the underlying normal distribution
        sigma: standard deviation of the underlying normal distribution
        min_val: minimum cost (constrain)
        max_val: maximum cost (constrain)
    
    Returns:
        A function that samples from the distribution
    """
    def sample(n=1):
        samples = np.random.lognormal(mu, sigma, size=n)
        # Constrain to range
        samples = np.clip(samples, min_val, max_val)
        return samples
    
    return sample

print("Custom distribution functions defined.")
print("These can be used to sample costs based on your own data or assumptions.")

In [ ]:
# Demonstrate custom distributions

# Create log-normal distribution with realistic parameters for residential upgrades
# mu and sigma chosen to give mean around $8,000
custom_lognormal = create_lognormal_distribution(mu=8.8, sigma=0.6, min_val=1000, max_val=50000)

# Sample from custom distribution
custom_samples = custom_lognormal(n=5000)

# Visualize
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=res_costs,
    nbinsx=50,
    name='BICEP Default (Residential)',
    opacity=0.6
))

fig.add_trace(go.Histogram(
    x=custom_samples,
    nbinsx=50,
    name='Custom Log-Normal',
    opacity=0.6
))

fig.update_layout(
    title='Comparing Default vs Custom Cost Distribution',
    xaxis_title='Cost ($)',
    yaxis_title='Frequency',
    barmode='overlay',
    height=500
)

fig.update_xaxes(tickformat='$,.0f')
fig.show()

print(f"\nCustom Log-Normal Distribution Statistics:")
print(f"  Mean: ${np.mean(custom_samples):,.0f}")
print(f"  Median: ${np.median(custom_samples):,.0f}")
print(f"  Std Dev: ${np.std(custom_samples):,.0f}")

## Impact of Distribution Choice on Total Costs

Different distributions can lead to significantly different total cost estimates.

In [ ]:
# Calculate impact of distribution choice

# Get building count that needs upgrades
num_buildings_with_upgrades = len(buildings_with_upgrades)

# Calculate total costs with different distributions
default_mean = np.mean(res_costs)
custom_mean = np.mean(custom_samples)

total_default = default_mean * num_buildings_with_upgrades
total_custom = custom_mean * num_buildings_with_upgrades

difference = total_custom - total_default
percent_diff = (difference / total_default) * 100

print(f"\nImpact of Distribution Choice on Total Costs:")
print(f"="*60)
print(f"Buildings needing upgrades: {num_buildings_with_upgrades:,}")
print(f"\nDefault Distribution (BICEP):")
print(f"  Mean cost per building: ${default_mean:,.0f}")
print(f"  Total estimated cost: ${total_default:,.0f}")
print(f"\nCustom Distribution (Log-Normal):")
print(f"  Mean cost per building: ${custom_mean:,.0f}")
print(f"  Total estimated cost: ${total_custom:,.0f}")
print(f"\nDifference:")
print(f"  Absolute: ${difference:,.0f}")
print(f"  Percent: {percent_diff:.1f}%")

## Key Takeaways

1. **Distributions capture uncertainty**: Costs aren't fixed; they vary based on many factors
2. **Distribution choice matters**: Different assumptions lead to significantly different estimates
3. **BICEP uses empirical and parametric distributions**: Based on real cost data and expert judgment
4. **Customization is possible**: You can implement your own distributions if needed
5. **Sensitivity analysis is important**: Test how results change with different cost assumptions

## Next Steps

- Review [Data Requirements](data-requirements.md) to understand technology adoption patterns
- Check [Scenario Comparison](scenario-comparison.md) for cost implications across scenarios
- Consult the [API Reference](../api-reference.md) for distribution class documentation